# YouTube Public Comment Scraper for the Ongoing Iran-US War

This notebook is separate from your news notebook and your other public-opinion notebook.

It uses the **YouTube Data API v3** with your API key to:
- search for war-related YouTube videos
- collect top-level public comments
- optionally collect replies to those comments
- keep only comments whose text or video context relates to the ongoing Iran-US war
- export both the comment dataset and the discovered video catalog

Quota note:
- `search.list` is the expensive step
- `commentThreads.list` and `comments.list` are much cheaper

Your API key is entered interactively in the notebook, so you do not need to hardcode it into the file.

The output of this scraper file is taken to Datasets/YT_Comments as `Youtube_Comments.csv`

>For the benefit of the scraper to work properly, please do not change any part of this scraper file

In [1]:
# Cell 1 - Install dependencies
%pip install -q pandas requests langdetect

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 - Imports and folders
import html
import json
import os
import re
import time
from datetime import datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from langdetect import DetectorFactory, LangDetectException, detect_langs
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# 1. Define the Output Directory
OUTPUT_DIR = Path(r'D:\COLLEGE\MBA\SEMESTERS\IV Semester\MBAG - 408 - TWA\24030302038_Master_Project\Datasets\Scraped Datasets\Youtube Comments')
OUTPUT_DIR.mkdir(exist_ok=True)

# 2. Check if the files already exist using a pattern search
existing_files = list(OUTPUT_DIR.glob("iran_us_youtube_comments_*.csv"))
ALREADY_SCRAPED = len(existing_files) > 0

if ALREADY_SCRAPED:
    print(f"✅ Already scraped. Found existing dataset: '{existing_files[0].name}'.")
    print(f"Delete the files in '{OUTPUT_DIR}' if you want to scrape again.")
else:
    print("No existing dataset found. Ready to scrape.")


✅ Already scraped. Found existing dataset: 'iran_us_youtube_comments_20260324_140423.csv'.
Delete the files in 'D:\COLLEGE\MBA\SEMESTERS\IV Semester\MBAG - 408 - TWA\24030302038_Master_Project\Datasets\Scraped Datasets\Youtube Comments' if you want to scrape again.


In [3]:
# Cell 3 - API key and research settings
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY", "").strip()
if not YOUTUBE_API_KEY:
    YOUTUBE_API_KEY = getpass("Enter your YouTube Data API v3 key: ").strip()
if not YOUTUBE_API_KEY:
    raise ValueError("A YouTube Data API key is required to continue.")

TOPIC_LABEL = "Public YouTube comments on the ongoing Iran-US war"
LOOKBACK_DAYS = 120
REQUEST_DELAY_SECONDS = 0.1
SEARCH_ORDER = "date"
RELEVANCE_LANGUAGE = "en"
REGION_CODE = None
MAX_VIDEOS_PER_QUERY = 30
MAX_TOP_LEVEL_COMMENTS_PER_VIDEO = 200
INCLUDE_REPLIES = True
MAX_REPLIES_PER_THREAD = 50
ONLY_KEEP_WAR_RELEVANT_COMMENTS = True
ONLY_KEEP_ENGLISH_COMMENTS = True
TARGET_COMMENT_LANGUAGE = "en"
MIN_COMMENT_CHARS_FOR_LANGUAGE_CHECK = 8
MIN_LANGUAGE_CONFIDENCE = 0.70

SEARCH_QUERIES = [
    "Iran US war",
    "Iran US conflict",
    "Iran US strikes",
    "Iran Israel US war",
    "Tehran Washington conflict",
    "Iran retaliation against US",
    "US attack on Iran",
    "Iran missile strike US Middle East",
]

WAR_TERM_GROUPS = {
    "iran": {
        "iran": r"\biran\b",
        "iranian": r"\biranian\b",
        "tehran": r"\btehran\b",
        "irgc": r"\birgc\b",
        "khamenei": r"\bkhamenei\b",
        "hormuz": r"\bhormuz\b",
        "persian gulf": r"\bpersian gulf\b",
    },
    "us": {
        "united states": r"\bunited states\b",
        "usa": r"\bu\.s\.a\b|\busa\b",
        "america": r"\bamerica\b|\bamerican\b",
        "washington": r"\bwashington\b",
        "white house": r"\bwhite house\b",
        "pentagon": r"\bpentagon\b",
        "centcom": r"\bcentcom\b",
        "trump": r"\btrump\b",
        "biden": r"\bbiden\b",
    },
    "war": {
        "war": r"\bwar\b",
        "conflict": r"\bconflict\b",
        "strike": r"\bstrike\b|\bstrikes\b",
        "attack": r"\battack\b|\battacks\b",
        "missile": r"\bmissile\b|\bmissiles\b",
        "drone": r"\bdrone\b|\bdrones\b",
        "retaliation": r"\bretaliat(?:e|ion)\b",
        "ceasefire": r"\bceasefire\b",
        "bombing": r"\bbombing\b|\bbombard(?:ed|ment)?\b",
        "military": r"\bmilitary\b",
    },
    "regional": {
        "israel": r"\bisrael\b|\bisraeli\b",
        "gaza": r"\bgaza\b",
        "hezbollah": r"\bhezbollah\b",
        "houthis": r"\bhouthis?\b",
        "middle east": r"\bmiddle east\b",
        "gulf": r"\bgulf\b",
        "iraq": r"\biraq\b",
        "syria": r"\bsyria\b",
        "lebanon": r"\blebanon\b",
        "saudi": r"\bsaudi\b",
        "uae": r"\buae\b|\bunited arab emirates\b",
    },
}

COMPILED_WAR_PATTERNS = {
    group: [(label, re.compile(pattern, re.IGNORECASE)) for label, pattern in patterns.items()]
    for group, patterns in WAR_TERM_GROUPS.items()
}

print("Queries configured           :", len(SEARCH_QUERIES))
print("Max videos per query         :", MAX_VIDEOS_PER_QUERY)
print("Max top-level comments/video :", MAX_TOP_LEVEL_COMMENTS_PER_VIDEO)
print("Include replies              :", INCLUDE_REPLIES)
print("English-only comments        :", ONLY_KEEP_ENGLISH_COMMENTS)
print("Target comment language      :", TARGET_COMMENT_LANGUAGE)
print("Quota note                   : search.list is the expensive API step")


ValueError: A YouTube Data API key is required to continue.

In [ ]:
# Cell 4 - Review the search queries
query_df = pd.DataFrame({"query": SEARCH_QUERIES})
display(query_df)

,query
0,Iran US war
1,Iran US conflict
2,Iran US strikes
3,Iran Israel US war
4,Tehran Washington conflict
5,Iran retaliation against US
6,US attack on Iran
7,Iran missile strike US Middle East


In [ ]:
# Cell 5 - Helper functions and YouTube API methods
YOUTUBE_API_BASE = "https://www.googleapis.com/youtube/v3"
DEFAULT_HEADERS = {
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
}


def build_http_session():
    session = requests.Session()
    retry_strategy = Retry(
        total=2,
        connect=2,
        read=2,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET", "HEAD"]),
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update(DEFAULT_HEADERS)
    return session


HTTP_SESSION = build_http_session()
DetectorFactory.seed = 0


def clean_text(text):
    text = html.unescape(text or "")
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def detect_comment_language(text):
    cleaned = clean_text(text)
    if len(cleaned) < MIN_COMMENT_CHARS_FOR_LANGUAGE_CHECK:
        return "unknown", 0.0

    try:
        ranked = detect_langs(cleaned)
    except LangDetectException:
        return "unknown", 0.0
    except Exception:
        return "unknown", 0.0

    if not ranked:
        return "unknown", 0.0

    best_match = ranked[0]
    return best_match.lang, float(best_match.prob)


def comment_passes_language_filter(text):
    detected_language, confidence = detect_comment_language(text)
    keep_comment = True

    if ONLY_KEEP_ENGLISH_COMMENTS:
        keep_comment = (
            detected_language == TARGET_COMMENT_LANGUAGE
            and confidence >= MIN_LANGUAGE_CONFIDENCE
        )

    return keep_comment, detected_language, confidence


def detect_war_context(text):
    cleaned = clean_text(text)
    matched = {}

    for group, patterns in COMPILED_WAR_PATTERNS.items():
        hits = sorted({label for label, pattern in patterns if pattern.search(cleaned)})
        if hits:
            matched[group] = hits

    score = sum(len(hits) for hits in matched.values())
    return matched, score


def evaluate_comment_context(comment_text, context_text=""):
    comment_matches, comment_score = detect_war_context(comment_text)
    context_matches, context_score = detect_war_context(context_text)

    def passes(match_map, score):
        has_iran = "iran" in match_map
        has_us = "us" in match_map
        has_war = "war" in match_map
        has_regional = "regional" in match_map
        return has_iran and (has_us or has_war or (has_regional and score >= 2) or score >= 4)

    comment_ok = passes(comment_matches, comment_score)
    context_ok = passes(context_matches, context_score)
    best_matches = comment_matches if comment_score >= context_score else context_matches
    best_score = max(comment_score, context_score)
    matched_terms = sorted({term for terms in best_matches.values() for term in terms})
    return comment_ok or context_ok, best_score, ", ".join(sorted(best_matches)), ", ".join(matched_terms), "comment" if comment_ok else "context"


def youtube_api_get(endpoint, params):
    payload = dict(params)
    payload["key"] = YOUTUBE_API_KEY
    response = HTTP_SESSION.get(f"{YOUTUBE_API_BASE}/{endpoint}", params=payload, timeout=30)

    if response.status_code >= 400:
        message = response.text
        try:
            error_payload = response.json()
            message = error_payload.get("error", {}).get("message", message)
        except Exception:
            pass
        raise RuntimeError(f"YouTube API error {response.status_code}: {message}")

    return response.json()


def search_videos(query, max_results=20):
    published_after = (datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)).isoformat(timespec="seconds").replace("+00:00", "Z")
    records = []
    seen_video_ids = set()
    page_token = None

    while len(records) < max_results:
        remaining = min(50, max_results - len(records))
        params = {
            "part": "snippet",
            "q": query,
            "type": "video",
            "order": SEARCH_ORDER,
            "publishedAfter": published_after,
            "maxResults": remaining,
        }
        if page_token:
            params["pageToken"] = page_token
        if RELEVANCE_LANGUAGE:
            params["relevanceLanguage"] = RELEVANCE_LANGUAGE
        if REGION_CODE:
            params["regionCode"] = REGION_CODE

        payload = youtube_api_get("search", params)

        for item in payload.get("items", []):
            video_id = item.get("id", {}).get("videoId", "")
            snippet = item.get("snippet", {})
            if not video_id or video_id in seen_video_ids:
                continue
            records.append(
                {
                    "query": query,
                    "video_id": video_id,
                    "video_title": clean_text(snippet.get("title", "")),
                    "video_description": clean_text(snippet.get("description", "")),
                    "channel_title": clean_text(snippet.get("channelTitle", "")),
                    "video_published_at": snippet.get("publishedAt", ""),
                    "video_url": f"https://www.youtube.com/watch?v={video_id}",
                }
            )
            seen_video_ids.add(video_id)
            if len(records) >= max_results:
                break

        page_token = payload.get("nextPageToken")
        if not page_token or len(records) >= max_results:
            break
        if REQUEST_DELAY_SECONDS > 0:
            time.sleep(REQUEST_DELAY_SECONDS)

    return records


def fetch_replies(parent_comment_id, max_results=50):
    replies = []
    seen_reply_ids = set()
    page_token = None

    while len(replies) < max_results:
        remaining = min(100, max_results - len(replies))
        params = {
            "part": "snippet",
            "parentId": parent_comment_id,
            "maxResults": remaining,
            "textFormat": "plainText",
        }
        if page_token:
            params["pageToken"] = page_token

        payload = youtube_api_get("comments", params)
        for item in payload.get("items", []):
            reply_id = item.get("id", "")
            if not reply_id or reply_id in seen_reply_ids:
                continue
            replies.append(item)
            seen_reply_ids.add(reply_id)
            if len(replies) >= max_results:
                break

        page_token = payload.get("nextPageToken")
        if not page_token or len(replies) >= max_results:
            break
        if REQUEST_DELAY_SECONDS > 0:
            time.sleep(REQUEST_DELAY_SECONDS)

    return replies


def build_comment_record(video, query, comment_id, parent_comment_id, thread_id, is_reply, author, text, like_count, published_at, updated_at, detected_language, language_confidence, context_score, matched_groups, matched_terms, kept_from):
    return {
        "topic": TOPIC_LABEL,
        "platform": "YouTube",
        "query": query,
        "video_id": video["video_id"],
        "video_title": video["video_title"],
        "video_description": video["video_description"],
        "video_url": video["video_url"],
        "channel_title": video["channel_title"],
        "video_published_at": video["video_published_at"],
        "comment_id": comment_id,
        "thread_id": thread_id,
        "parent_comment_id": parent_comment_id,
        "is_reply": is_reply,
        "author": clean_text(author),
        "comment_text": clean_text(text),
        "detected_language": detected_language,
        "language_confidence": language_confidence,
        "like_count": like_count,
        "published_at": published_at,
        "updated_at": updated_at,
        "context_score": context_score,
        "matched_groups": matched_groups,
        "matched_terms": matched_terms,
        "kept_from": kept_from,
        "collection_method": "YouTube Data API v3",
    }


def collect_comments_for_video(video, query):
    records = []
    seen_comment_ids = set()
    page_token = None
    top_level_seen = 0
    video_context = f"{video['video_title']} {video['video_description']} {query}"

    while top_level_seen < MAX_TOP_LEVEL_COMMENTS_PER_VIDEO:
        remaining_top_level = min(100, MAX_TOP_LEVEL_COMMENTS_PER_VIDEO - top_level_seen)
        params = {
            "part": "snippet,replies",
            "videoId": video["video_id"],
            "maxResults": remaining_top_level,
            "order": "time",
            "textFormat": "plainText",
        }
        if page_token:
            params["pageToken"] = page_token

        payload = youtube_api_get("commentThreads", params)

        for thread in payload.get("items", []):
            thread_id = thread.get("id", "")
            thread_snippet = thread.get("snippet", {})
            top_level_comment = thread_snippet.get("topLevelComment", {})
            top_level_id = top_level_comment.get("id", "")
            top_level_snippet = top_level_comment.get("snippet", {})
            top_level_text = clean_text(top_level_snippet.get("textDisplay", ""))
            language_ok, detected_language, language_confidence = comment_passes_language_filter(top_level_text)
            keep_item, context_score, matched_groups, matched_terms, kept_from = evaluate_comment_context(top_level_text, video_context)

            if (
                language_ok
                and (not ONLY_KEEP_WAR_RELEVANT_COMMENTS or keep_item)
                and top_level_id
                and top_level_id not in seen_comment_ids
            ):
                records.append(
                    build_comment_record(
                        video=video,
                        query=query,
                        comment_id=top_level_id,
                        parent_comment_id="",
                        thread_id=thread_id,
                        is_reply=False,
                        author=top_level_snippet.get("authorDisplayName", ""),
                        text=top_level_text,
                        like_count=top_level_snippet.get("likeCount", 0),
                        published_at=top_level_snippet.get("publishedAt", ""),
                        updated_at=top_level_snippet.get("updatedAt", ""),
                        detected_language=detected_language,
                        language_confidence=language_confidence,
                        context_score=context_score,
                        matched_groups=matched_groups,
                        matched_terms=matched_terms,
                        kept_from=kept_from,
                    )
                )
                seen_comment_ids.add(top_level_id)

            top_level_seen += 1

            if INCLUDE_REPLIES and top_level_id:
                embedded_replies = list(thread.get("replies", {}).get("comments", []) or [])
                total_reply_count = int(thread_snippet.get("totalReplyCount", 0) or 0)
                max_replies_needed = min(total_reply_count, MAX_REPLIES_PER_THREAD)

                if max_replies_needed > len(embedded_replies):
                    try:
                        extra_replies = fetch_replies(top_level_id, max_replies_needed)
                    except Exception:
                        extra_replies = embedded_replies
                else:
                    extra_replies = embedded_replies

                reply_pool = []
                reply_seen = set()
                for reply in embedded_replies + extra_replies:
                    reply_id = reply.get("id", "")
                    if not reply_id or reply_id in reply_seen:
                        continue
                    reply_pool.append(reply)
                    reply_seen.add(reply_id)
                    if len(reply_pool) >= MAX_REPLIES_PER_THREAD:
                        break

                for reply in reply_pool:
                    reply_id = reply.get("id", "")
                    reply_snippet = reply.get("snippet", {})
                    reply_text = clean_text(reply_snippet.get("textDisplay", ""))
                    language_ok, detected_language, language_confidence = comment_passes_language_filter(reply_text)
                    keep_reply, reply_score, reply_groups, reply_terms, reply_kept_from = evaluate_comment_context(reply_text, video_context)
                    if ONLY_KEEP_ENGLISH_COMMENTS and not language_ok:
                        continue
                    if ONLY_KEEP_WAR_RELEVANT_COMMENTS and not keep_reply:
                        continue
                    if reply_id in seen_comment_ids:
                        continue
                    records.append(
                        build_comment_record(
                            video=video,
                            query=query,
                            comment_id=reply_id,
                            parent_comment_id=top_level_id,
                            thread_id=thread_id,
                            is_reply=True,
                            author=reply_snippet.get("authorDisplayName", ""),
                            text=reply_text,
                            like_count=reply_snippet.get("likeCount", 0),
                            published_at=reply_snippet.get("publishedAt", ""),
                            updated_at=reply_snippet.get("updatedAt", ""),
                            detected_language=detected_language,
                            language_confidence=language_confidence,
                            context_score=reply_score,
                            matched_groups=reply_groups,
                            matched_terms=reply_terms,
                            kept_from=reply_kept_from,
                        )
                    )
                    seen_comment_ids.add(reply_id)

        page_token = payload.get("nextPageToken")
        if not page_token or top_level_seen >= MAX_TOP_LEVEL_COMMENTS_PER_VIDEO:
            break
        if REQUEST_DELAY_SECONDS > 0:
            time.sleep(REQUEST_DELAY_SECONDS)

    return records


COMMENT_COLUMNS = [
    "topic",
    "platform",
    "query",
    "video_id",
    "video_title",
    "video_description",
    "video_url",
    "channel_title",
    "video_published_at",
    "comment_id",
    "thread_id",
    "parent_comment_id",
    "is_reply",
    "author",
    "comment_text",
    "detected_language",
    "language_confidence",
    "like_count",
    "published_at",
    "updated_at",
    "context_score",
    "matched_groups",
    "matched_terms",
    "kept_from",
    "collection_method",
]

VIDEO_COLUMNS = [
    "query",
    "video_id",
    "video_title",
    "video_description",
    "channel_title",
    "video_published_at",
    "video_url",
]


def collect_youtube_public_comments():
    all_records = []
    video_catalog = []
    seen_video_ids = set()
    seen_comment_ids = set()

    for query in SEARCH_QUERIES:
        print("=" * 90)
        print(f"Query: {query}")
        try:
            videos = search_videos(query, max_results=MAX_VIDEOS_PER_QUERY)
            print(f"Videos found: {len(videos)}")
        except Exception as exc:
            print(f"Video search failed: {exc}")
            continue

        for video in videos:
            if video["video_id"] in seen_video_ids:
                continue
            seen_video_ids.add(video["video_id"])
            video_catalog.append(video)
            print(f"Collecting comments from: {video['video_title'][:100]}")
            try:
                video_records = collect_comments_for_video(video, query)
                kept_now = 0
                for record in video_records:
                    if record["comment_id"] in seen_comment_ids:
                        continue
                    all_records.append(record)
                    seen_comment_ids.add(record["comment_id"])
                    kept_now += 1
                print(f"Kept comments: {kept_now}")
            except Exception as exc:
                print(f"Comment collection failed for video {video['video_id']}: {exc}")
                continue

    comments_df = pd.DataFrame(all_records, columns=COMMENT_COLUMNS) if all_records else pd.DataFrame(columns=COMMENT_COLUMNS)
    videos_df = pd.DataFrame(video_catalog, columns=VIDEO_COLUMNS) if video_catalog else pd.DataFrame(columns=VIDEO_COLUMNS)

    if not comments_df.empty:
        comments_df = comments_df.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)
        comments_df = comments_df.sort_values(["published_at", "video_title"], ascending=[False, True], na_position="last").reset_index(drop=True)

    if not videos_df.empty:
        videos_df = videos_df.drop_duplicates(subset=["video_id"]).reset_index(drop=True)
        videos_df = videos_df.sort_values(["video_published_at", "video_title"], ascending=[False, True], na_position="last").reset_index(drop=True)

    return comments_df, videos_df


In [ ]:
# Cell 6 - Main collection runner

if ALREADY_SCRAPED:
    print("Skipping YouTube collection... Dataset already exists in your folder.")
    
    # Load the existing comments data into memory
    youtube_comments_df = pd.read_csv(existing_files[0])
    
    # Automatically find and load the matching videos catalog file
    video_file_name = existing_files[0].name.replace("comments", "videos")
    video_file_path = OUTPUT_DIR / video_file_name
    if video_file_path.exists():
        youtube_videos_df = pd.read_csv(video_file_path)
    else:
        youtube_videos_df = pd.DataFrame()
        
else:
    print("Starting YouTube Data API collection...")
    # This is where your original heavy scraping function runs
    youtube_comments_df, youtube_videos_df = collect_youtube_public_comments()

print("Total comments kept:", len(youtube_comments_df))
print("Unique videos      :", len(youtube_videos_df))
print("Unique channels    :", youtube_comments_df["channel_title"].nunique() if not youtube_comments_df.empty else 0)

if youtube_comments_df.empty:
    print("No comments were collected. Check the API key, quota, YouTube comments availability, or widen the search queries.")
else:
    source_summary_df = youtube_comments_df.groupby(["query", "channel_title"]).size().reset_index(name="row_count").sort_values(["row_count", "query"], ascending=[False, True])
    preview_df = youtube_comments_df[["query", "video_title", "author", "comment_text", "detected_language", "language_confidence", "is_reply", "context_score"]].head(30)
    display(source_summary_df.head(40))
    display(preview_df)

Skipping YouTube collection... Dataset already exists in your folder.
Total comments kept: 2131
Unique videos      : 208
Unique channels    : 52


,query,channel_title,row_count
42,Iran retaliation against US,The Military Show,320
61,US attack on Iran,New York Post,233
22,Iran US war,BBC News,202
36,Iran retaliation against US,FOX 10 Phoenix,196
30,Iran missile strike US Middle East,TBN Israel,187
21,Iran US war,13WHAM ABC News,186
12,Iran US strikes,KCENNews,172
53,Tehran Washington conflict,Times Now World,101
50,Tehran Washington conflict,Hindustan Times,59
31,Iran missile strike US Middle East,WION,43


,query,video_title,author,comment_text,detected_language,language_confidence,is_reply,context_score
0,Iran missile strike US Middle East,BREAKING: U.S. Halts DEVASTATING Military Stri...,@woopywoopydodo,Trump is useing taqiyya on them😂😂. No way woul...,en,0.999998,False,8
1,Iran US strikes,US-Iran War: Iran Fires Missiles Towards North...,@myfitnessnutrition1104,First all world's common peoples should be uni...,en,0.999997,False,6
2,Iran US strikes,US Hits Iran Gas Sites! Iran Strikes AMAZON & ...,@Aadi_remover_of_sorrow,Bangladesh closely monitoring the situation 😂,en,0.999994,True,3
3,Iran missile strike US Middle East,BREAKING: U.S. Halts DEVASTATING Military Stri...,@lesterschulze5266,Thank you Brothers Yair and Marti for keeping ...,en,0.999997,False,8
4,Iran US strikes,US Hits Iran Gas Sites! Iran Strikes AMAZON & ...,@ShubhamMishra-yw8nu,Usa should stop ths fight,en,0.999997,False,3
5,Iran US war,"RAW: Trump shares update on Iran war efforts, ...",@amandaharwell2017,He ask why didn't they capture and he said the...,en,0.999995,False,5
6,US attack on Iran,"Israel, US attack Iran: ट्रंप के महायुद्ध से ब...",@freekagyaan72,Mossad behind this refinery blast,en,0.999996,False,3
7,Iran retaliation against US,"🟢Israel Iran War LIVE: Iran Retaliates To US, ...",@jewaypakistan8551,It means dt China & Russia r helping Iran. Wha...,en,0.999997,False,10
8,Iran US strikes,US Hits Iran Gas Sites! Iran Strikes AMAZON & ...,@creativearch7530,Fix it.... otherwise don't say ki you are prov...,en,0.999998,False,3
9,Iran US strikes,US Hits Iran Gas Sites! Iran Strikes AMAZON & ...,@newbharat731,After 6 minutes let's discuss it in next 4 min...,en,0.857139,False,3


In [ ]:
# Cell 7 - Total comments gathered
if youtube_comments_df.empty:
    print('Total comments gathered: 0')
    print('Top-level comments    : 0')
    print('Replies               : 0')
    print('Unique videos covered : 0')
else:
    total_comments = len(youtube_comments_df)
    top_level_comments = int((~youtube_comments_df['is_reply']).sum())
    reply_comments = int(youtube_comments_df['is_reply'].sum())
    unique_videos = youtube_comments_df['video_id'].nunique()

    print(f'Total comments gathered: {total_comments}')
    print(f'Top-level comments    : {top_level_comments}')
    print(f'Replies               : {reply_comments}')
    print(f'Unique videos covered : {unique_videos}')


Total comments gathered: 2131
Top-level comments    : 1632
Replies               : 499
Unique videos covered : 73


In [ ]:
# Cell 8 - Export the dataset
if ALREADY_SCRAPED:
    print("Skipping export. Your YouTube data is already safely stored in your folder!")
else:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    comments_csv_path = OUTPUT_DIR / f"iran_us_youtube_comments_{timestamp}.csv"
    comments_json_path = OUTPUT_DIR / f"iran_us_youtube_comments_{timestamp}.json"
    videos_csv_path = OUTPUT_DIR / f"iran_us_youtube_videos_{timestamp}.csv"
    videos_json_path = OUTPUT_DIR / f"iran_us_youtube_videos_{timestamp}.json"
    settings_path = OUTPUT_DIR / f"iran_us_youtube_settings_{timestamp}.json"

    youtube_comments_df.to_csv(comments_csv_path, index=False, encoding="utf-8-sig")
    youtube_comments_df.to_json(comments_json_path, orient="records", indent=2, force_ascii=False)
    youtube_videos_df.to_csv(videos_csv_path, index=False, encoding="utf-8-sig")
    youtube_videos_df.to_json(videos_json_path, orient="records", indent=2, force_ascii=False)
    
    settings_path.write_text(
        json.dumps(
            {
                "topic": TOPIC_LABEL,
                "lookback_days": LOOKBACK_DAYS,
                "search_order": SEARCH_ORDER,
                "relevance_language": RELEVANCE_LANGUAGE,
                "region_code": REGION_CODE,
                "only_keep_english_comments": ONLY_KEEP_ENGLISH_COMMENTS,
                "target_comment_language": TARGET_COMMENT_LANGUAGE,
                "min_comment_chars_for_language_check": MIN_COMMENT_CHARS_FOR_LANGUAGE_CHECK,
                "min_language_confidence": MIN_LANGUAGE_CONFIDENCE,
                "max_videos_per_query": MAX_VIDEOS_PER_QUERY,
                "max_top_level_comments_per_video": MAX_TOP_LEVEL_COMMENTS_PER_VIDEO,
                "include_replies": INCLUDE_REPLIES,
                "max_replies_per_thread": MAX_REPLIES_PER_THREAD,
                "queries": SEARCH_QUERIES,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    print("Comments CSV saved to:", comments_csv_path.resolve())
    print("Comments JSON saved to:", comments_json_path.resolve())
    print("Videos CSV saved to  :", videos_csv_path.resolve())
    print("Videos JSON saved to :", videos_json_path.resolve())
    print("Settings saved to    :", settings_path.resolve())

Skipping export. Your YouTube data is already safely stored in your folder!
